In [ ]:
# ПРОЕКТ: РАСПОЗНАВАНИЕ ЭМОЦИЙ

# 1. УСТАНОВКА И ИМПОРТ БИБЛИОТЕК

!pip install opencv-python-headless scikit-learn seaborn matplotlib numpy tensorflow

import os
import zipfile
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
from collections import Counter
from sklearn.metrics import confusion_matrix, classification_report
from sklearn.utils.class_weight import compute_class_weight

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.preprocessing import image

# Настройки для графиков
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["axes.grid"] = False

print("TensorFlow:", tf.__version__)
print("NumPy:", np.__version__)
print("OpenCV:", cv2.__version__)
print("Все библиотеки загружены!")



# 2. ЗАГРУЗКА И РАСПАКОВКА ДАТАСЕТА FER2013

DATA_PATH = "/content/fer2013_data"


print("Нажмите на кнопку 'Choose Files' и выбери archive.zip с датасетом FER2013")
print("Скачать датасет можно здесь: https://www.kaggle.com/datasets/msambare/fer2013")

uploaded = files.upload()

# Берём первый загруженный файл
zip_filename = list(uploaded.keys())[0]
print(f"\nЗагружено: {zip_filename}")

# Распаковываем
with zipfile.ZipFile(zip_filename, 'r') as zip_ref:
    zip_ref.extractall(DATA_PATH)

print(f"\nДатасет распакован в {DATA_PATH}")
print("Содержимое:", os.listdir(DATA_PATH))

# Проверяем структуру
train_path = os.path.join(DATA_PATH, 'train')
if os.path.exists(train_path):
    classes = sorted(os.listdir(train_path))
    print("\nКлассы эмоций:", classes)
    print("Количество классов:", len(classes))

    # Считаем картинки в каждой папке
    print("\nРаспределение изображений по классам:")
    for cls in classes:
        cls_path = os.path.join(train_path, cls)
        count = len(os.listdir(cls_path))
        print(f"  {cls:12s}: {count} изображений")
else:
    print("Ошибка: структура папок не соответствует ожидаемой")
    print("Содержимое:", os.listdir(DATA_PATH))



# 3. ПАРАМЕТРЫ И ЗАГРУЗКА ДАННЫХ


IMG_SIZE = 48
BATCH_SIZE = 64

# Пути к папкам
train_dir = os.path.join(DATA_PATH, "train")
val_dir = os.path.join(DATA_PATH, "test")  # В FER2013 папка test - это валидация

# Классы эмоций
class_names = sorted(os.listdir(train_dir))
print(f"\nКлассы эмоций: {class_names}")
print(f"Всего классов: {len(class_names)}")

# Загружаем тренировочные данные (с валидационным сплитом)
train_ds = keras.utils.image_dataset_from_directory(
    train_dir,
    labels='inferred',
    label_mode='int',
    image_size=(IMG_SIZE, IMG_SIZE),
    color_mode='grayscale',  # FER2013 - grayscale изображения
    batch_size=BATCH_SIZE,
    shuffle=True,
    validation_split=0.2,
    subset='training',
    seed=42
)

val_ds = keras.utils.image_dataset_from_directory(
    train_dir,
    labels='inferred',
    label_mode='int',
    image_size=(IMG_SIZE, IMG_SIZE),
    color_mode='grayscale',
    batch_size=BATCH_SIZE,
    shuffle=True,
    validation_split=0.2,
    subset='validation',
    seed=42
)

# Оптимизация производительности
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(AUTOTUNE)
val_ds = val_ds.prefetch(AUTOTUNE)

print(f"\nДанные загружены!")
print(f"Размер изображения: {IMG_SIZE}x{IMG_SIZE} (grayscale)")



# 4. ВИЗУАЛИЗАЦИЯ ДАННЫХ


plt.figure(figsize=(12, 8))

for images, labels in train_ds.take(1):
    for i in range(12):
        ax = plt.subplot(3, 4, i + 1)
        # Изображения уже grayscale, отображаем как есть
        plt.imshow(images[i].numpy().squeeze(), cmap='gray')
        plt.title(class_names[labels[i]])
        plt.axis("off")

plt.suptitle("Примеры изображений из датасета", fontsize=14)
plt.tight_layout()
plt.show()







In [ ]:
# 5. СОЗДАНИЕ МОДЕЛИ CNN


def create_improved_model():
    """Создает улучшенную CNN модель с меньшим переобучением"""
    model = keras.Sequential([
        # Вход для grayscale (1 канал)
        layers.Input(shape=(IMG_SIZE, IMG_SIZE, 1)),
        layers.Rescaling(1./255),

        # Блок 1
        layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.Conv2D(64, (3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.3),

        # Блок 2
        layers.Conv2D(128, (3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.Conv2D(128, (3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.4),

        # Блок 3
        layers.Conv2D(256, (3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.Conv2D(256, (3, 3), activation='relu', padding='same'),
        layers.BatchNormalization(),
        layers.GlobalAveragePooling2D(),  # Вместо Flatten для уменьшения параметров
        layers.Dropout(0.5),

        # Полносвязная часть
        layers.Dense(512, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.5),
        layers.Dense(256, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.3),
        layers.Dense(len(class_names), activation='softmax')
    ])
    return model

# Создаем модель
model = create_improved_model()

# Выводим архитектуру
print("Архитектура модели:")
model.summary()



# 6. КОМПИЛЯЦИЯ МОДЕЛИ

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print("Модель скомпилирована!")


In [ ]:
#  ОБУЧЕНИЕ МОДЕЛИ

# Параметры обучения
EPOCHS = 12

# Ранняя остановка - автоматически остановит обучение, если результат не улучшается
early_stopping = keras.callbacks.EarlyStopping(
    monitor='val_accuracy',  # Следим за точностью на валидации
    patience=3,              # Если 3 эпохи подряд нет улучшения - останавливаем
    restore_best_weights=True,  # Возвращаем веса лучшей эпохи
    verbose=1                # Печатаем сообщение об остановке
)

print("Начинаю обучение модели...")
print(f"Максимум эпох: {EPOCHS}")
print("Обучение остановится раньше, если точность перестанет расти")

# Обучаем модель
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=[early_stopping],  # Добавляем раннюю остановку
    verbose=1
)

print("Обучение завершено!")

In [ ]:

# 8. ГРАФИКИ ОБУЧЕНИЯ

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# График ошибки (loss)
axes[0].plot(history.history['loss'], label='Train Loss', linewidth=2)
axes[0].plot(history.history['val_loss'], label='Validation Loss', linewidth=2)
axes[0].set_title('Ошибка (Loss) во время обучения', fontsize=12)
axes[0].set_xlabel('Эпоха')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# График точности (accuracy)
axes[1].plot(history.history['accuracy'], label='Train Accuracy', linewidth=2)
axes[1].plot(history.history['val_accuracy'], label='Validation Accuracy', linewidth=2)
axes[1].set_title('Точность (Accuracy) во время обучения', fontsize=12)
axes[1].set_xlabel('Эпоха')
axes[1].set_ylabel('Accuracy')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('Динамика обучения модели', fontsize=14)
plt.tight_layout()
plt.show()



# 9. ФИНАЛЬНАЯ ОЦЕНКА НА ВАЛИДАЦИИ

val_loss, val_acc = model.evaluate(val_ds, verbose=0)
print(f"\nРезультаты на валидационной выборке:")
print(f"Точность: {val_acc:.4f} ({val_acc*100:.2f}%)")
print(f"Ошибка: {val_loss:.4f}")


# ----------------------------------------
# 10. МАТРИЦА ОШИБОК И ОТЧЕТ ПО КЛАССАМ
# ----------------------------------------

# Получаем предсказания
y_true = []
y_pred = []

for images, labels in val_ds:
    preds = model.predict(images, verbose=0)
    y_true.extend(labels.numpy())
    y_pred.extend(np.argmax(preds, axis=1))

# Строим confusion matrix
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.title('Матрица ошибок (Confusion Matrix)', fontsize=14)
plt.xlabel('Предсказанная эмоция', fontsize=12)
plt.ylabel('Истинная эмоция', fontsize=12)
plt.tight_layout()
plt.show()

# Детальный отчёт
print("\nОтчёт по каждому классу:")
print(classification_report(y_true, y_pred, target_names=class_names))




In [ ]:
# Дообучение модели. Улучшения только для disgust и fear (которых мало в датасете)
from sklearn.utils.class_weight import compute_class_weight

# Считаем веса классов (больше вес = меньше данных)
all_labels = []
for images, labels in train_ds:
    all_labels.extend(labels.numpy())

class_weights = compute_class_weight('balanced', classes=np.unique(all_labels), y=all_labels)
class_weight_dict = dict(enumerate(class_weights))

print("Веса классов (у disgust и fear самые большие):")
for i, name in enumerate(class_names):
    print(f"  {name:10s}: {class_weight_dict[i]:.3f}")

# Дообучаем 3 эпохи с весами
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.0001),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print("\nДообучаем 3 эпохи с балансировкой...")
history_fix = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=3,
    class_weight=class_weight_dict,
    verbose=1
)

# Проверяем новую точность
new_loss, new_acc = model.evaluate(val_ds, verbose=0)
print(f"\n📈 Точность была: 59%, стала: {new_acc:.1%}")

In [ ]:
# ФИНАЛЬНАЯ ДЕМОНСТРАЦИЯ ДЛЯ ПРОЕКТА (НА ДАННЫХ ИЗ ДАТАСЕТА)

import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

# 1. ОСНОВНЫЕ МЕТРИКИ
print("\nОСНОВНЫЕ РЕЗУЛЬТАТЫ:")

print(f"Точность модели: 61%")
print(f"Количество классов: {len(class_names)}")
print(f"Эмоции: {', '.join(class_names)}")

# 2. ДЕМОНСТРАЦИЯ НА ПРИМЕРАХ ИЗ ДАТАСЕТА
print("\n\nДЕМОНСТРАЦИЯ РАБОТЫ МОДЕЛИ (на данных из датасета):")

# Берем случайные примеры из валидации
sample_images, sample_labels = next(iter(val_ds.take(1)))
predictions = model.predict(sample_images[:12], verbose=0)

fig, axes = plt.subplots(3, 4, figsize=(14, 10))
fig.suptitle('Примеры распознавания эмоций моделью', fontsize=16, fontweight='bold')

for i in range(12):
    row = i // 4
    col = i % 4

    # Показываем фото
    axes[row, col].imshow(sample_images[i].numpy().squeeze(), cmap='gray')

    true_emotion = class_names[sample_labels[i]]
    pred_emotion = class_names[np.argmax(predictions[i])]
    confidence = np.max(predictions[i])

    # Цвет: зеленый - правильно, красный - ошибка
    color = 'green' if true_emotion == pred_emotion else 'red'

    axes[row, col].set_title(
        f"Правда: {true_emotion}\nПредсказано: {pred_emotion}\nУверенность: {confidence:.0%}",
        color=color, fontsize=9
    )
    axes[row, col].axis('off')

plt.tight_layout()
plt.show()

# 3. МАТРИЦА ОШИБОК
print("\n\nМАТРИЦА ОШИБОК (Confusion Matrix):")
print("-" * 50)

# Полная матрица ошибок
y_true_full = []
y_pred_full = []
for images, labels in val_ds:
    preds = model.predict(images, verbose=0)
    y_true_full.extend(labels.numpy())
    y_pred_full.extend(np.argmax(preds, axis=1))

cm = confusion_matrix(y_true_full, y_pred_full)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names,
            annot_kws={'size': 10})
plt.title('Матрица ошибок модели', fontsize=14, fontweight='bold')
plt.xlabel('Предсказанная эмоция', fontsize=12)
plt.ylabel('Истинная эмоция', fontsize=12)
plt.tight_layout()
plt.show()

# 4. ДЕТАЛЬНЫЙ ОТЧЕТ
print("\n\nДЕТАЛЬНЫЙ ОТЧЕТ ПО КАЖДОМУ КЛАССУ:")
print("-" * 50)

from sklearn.metrics import classification_report
report = classification_report(y_true_full, y_pred_full, target_names=class_names)
print(report)

# 6. ИТОГОВАЯ СТАТИСТИКА
from collections import Counter
pred_distribution = Counter([class_names[p] for p in y_pred_full])

print(f"\nОбщая точность модели: 61%")
print(f"  📊 Количество тестовых примеров: {len(y_true_full)}")
print(f"\nРаспределение предсказаний модели:")
for emotion, count in pred_distribution.items():
    pct = count/len(y_pred_full)*100
    bar = "█" * int(pct / 2)
    print(f"     {emotion:12s}: {count:4d} ({pct:5.1f}%) {bar}")

In [ ]:
# СОХРАНЕНИЕ МОДЕЛИ ДЛЯ СДАЧИ ПРОЕКТА

import json
from datetime import datetime

# 1. Сохраняем модель
model.save('/content/emotion_model_final.h5')
print("Модель сохранена как 'emotion_model_final.h5'")

# 2. Сохраняем метрики и результаты

# Финальные метрики
metrics = {
    'accuracy': 0.61,
    'classes': class_names,
    'dataset': 'FER2013',
    'image_size': '48x48 grayscale',
    'architecture': 'CNN with fine-tuning'
}

with open('/content/model_metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)
print("Метрики сохранены")

# 3. README

readme_content = f"""# Распознавание эмоций на лицах (Emotion Recognition)

##О проекте
Нейронная сеть для распознавания 7 эмоций на изображениях лиц:
- angry (злость)
- disgust (отвращение)
- fear (страх)
- happy (счастье)
- neutral (нейтральное)
- sad (грусть)
- surprise (удивление)

##Результаты
- **Точность модели: 61%**
- Датасет: FER2013 (35,000 изображений 48x48 grayscale)
- Архитектура: CNN + дообучение

##Файлы в репозитории
- `emotion_recognition.ipynb` - Jupyter notebook с кодом
- `emotion_model_final.h5` - обученная модель
- `model_metrics.json` - метрики модели

##Запуск
1. Открыть notebook в Google Colab
2. Загрузить датасет FER2013
3. Запустить все ячейки (последняя ячейка как раз проверяет модель на данных из датасета)

##Выводы
Модель успешно распознает эмоции на тестовых данных датасета.
Основные ошибки связаны с визуальной схожестью эмоций (fear/surprise)
и недостатком данных (disgust). С загруженными пользователем фотографиями модель справляется ещё хуже :(
"""

with open('/content/README.md', 'w', encoding='utf-8') as f:
    f.write(readme_content)
print("README.md создан")

# 4. Скачиваем все файлы на компуктер
from google.colab import files

# Скачиваем модель
files.download('/content/emotion_model_final.h5')
print(" есть emotion_model_final.h5")

# Скачиваем метрики
files.download('/content/model_metrics.json')
print("есть model_metrics.json")

# Скачиваем README
files.download('/content/README.md')
print("есть README.md")


print("ГОТОВО  ФАЙЛЫ СКАЧАНЫ")
